# Projeto 8 - Inteligência Financeira - Prevendo Preços de Ativos em Tempo Real

### Objetivo do Projeto: 
<p>Criar uma aplicação que possa fazer analise de ativos financeiros e gerar previsões em tempo real

In [ ]:
# Instalando o pacote yfinance que é um pacote oficial do Yahoo Finance para baixar dados atualizados de ações
#!pip install yfinance

# Instalando o pacote ta (Technical Analysis) que é um pacote onde podemos utilizar as funções prontas para calcular indicadores financeiros de ações
#!pip install ta

# Site do Yahoo Finance para ver as empresas e nomes de ativos
# https://finance.yahoo.com/markets/stocks/most-active/

In [1]:
! pip install yfinance


     ---------------------------------------- 0.0/144.1 kB ? eta -:--:--
     -------- ------------------------------ 30.7/144.1 kB 1.3 MB/s eta 0:00:01
     ------------------------ -------------- 92.2/144.1 kB 1.3 MB/s eta 0:00:01
     -------------------------------------- 144.1/144.1 kB 1.2 MB/s eta 0:00:00
     ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
     --- ------------------------------------ 0.2/2.0 MB 9.6 MB/s eta 0:00:01
     -------- ------------------------------- 0.4/2.0 MB 6.3 MB/s eta 0:00:01
     ------------------ --------------------- 0.9/2.0 MB 7.3 MB/s eta 0:00:01
     ---------------------------------------  1.9/2.0 MB 11.3 MB/s eta 0:00:01
     ---------------------------------------- 2.0/2.0 MB 10.5 MB/s eta 0:00:00
     ---------------------------------------- 0.0/109.9 kB ? eta -:--:--
     ---------------------------------------- 109.9/109.9 kB ? eta 0:00:00
     ---------------------------------------- 0.0/180.2 kB ? eta -:--:--
   


[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
! pip install ta

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Running setup.py install for ta: started
  Running setup.py install for ta: finished with status 'done'


  DEPRECATION: ta is being installed using the legacy 'setup.py install' method, because it does not have a 'pyproject.toml' and the 'wheel' package is not installed. pip 23.1 will enforce this behaviour change. A possible replacement is to enable the '--use-pep517' option. Discussion can be found at https://github.com/pypa/pip/issues/8559

[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import datetime
import pandas as pd
#import streamlit as st
import yfinance as yf
from datetime import date
from ta.momentum import RSIIndicator
from ta.volatility import BollingerBands
from ta.trend import MACD, EMAIndicator, SMAIndicator
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

In [2]:
# Criando uma função para baixar os dados de ativos
def download_dados(ticker, start_date, end_date):
    
    ativo = yf.Ticker(ticker)
    df = ativo.history(start=start_date, end=end_date)

    # Retorna o DataFrame com os dados baixados
    return df

In [3]:
# Definindo os parametros
num_pontos_dados = 720
today = datetime.date.today()
start_date = today - datetime.timedelta(days = num_pontos_dados)
end_date = today
stock = 'GOOG'

In [4]:
# Visualizando os valores dos parametros
print(num_pontos_dados)
print(today)
print(start_date)
print(end_date)
print(stock)

720
2026-08-12
2024-08-22
2026-08-12
GOOG


In [5]:
# Executando função que ira baixar os dados da api
df_dados = download_dados(stock,start_date,end_date)
df_dados.head()

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2024-08-22 00:00:00-04:00,167.753329,168.130441,163.773857,164.230362,19123800,0.0,0.0
2024-08-23 00:00:00-04:00,165.282287,166.671625,164.399062,166.155579,14281600,0.0,0.0
2024-08-26 00:00:00-04:00,166.875073,168.090754,165.054048,166.651779,11990300,0.0,0.0
2024-08-27 00:00:00-04:00,166.334189,166.964350,164.895229,165.113556,13718200,0.0,0.0
2024-08-28 00:00:00-04:00,165.510508,166.115865,162.037149,163.247864,15208700,0.0,0.0


In [6]:
# Calcula as Bandas de Bollinger para o preço de fechamento
indicador_bb = BollingerBands(df_dados.Close)
indicador_bb.bollinger_hband()

Date
2024-08-22 00:00:00-04:00           NaN
2024-08-23 00:00:00-04:00           NaN
2024-08-26 00:00:00-04:00           NaN
2024-08-27 00:00:00-04:00           NaN
2024-08-28 00:00:00-04:00           NaN
                                ...    
2026-08-05 00:00:00-04:00    380.005147
2026-08-06 00:00:00-04:00    380.044178
2026-08-07 00:00:00-04:00    379.904848
2026-08-10 00:00:00-04:00    380.331277
2026-08-11 00:00:00-04:00    379.404842
Name: hband, Length: 493, dtype: float64

In [7]:
# Faz uma cópia do DataFrame
df_bb = df_dados
df_bb.head()

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2024-08-22 00:00:00-04:00,167.753329,168.130441,163.773857,164.230362,19123800,0.0,0.0
2024-08-23 00:00:00-04:00,165.282287,166.671625,164.399062,166.155579,14281600,0.0,0.0
2024-08-26 00:00:00-04:00,166.875073,168.090754,165.054048,166.651779,11990300,0.0,0.0
2024-08-27 00:00:00-04:00,166.334189,166.964350,164.895229,165.113556,13718200,0.0,0.0
2024-08-28 00:00:00-04:00,165.510508,166.115865,162.037149,163.247864,15208700,0.0,0.0


In [10]:
# Insere uma boluna com o valor da banda de bolinger
df_bb['bb_maximo'] = indicador_bb.bollinger_hband()
df_bb.head(30)

,Open,High,Low,Close,Volume,Dividends,Stock Splits,bb_minimo,bb_maximo
Date,,,,,,,,,
2024-08-22 00:00:00-04:00,167.753329,168.130441,163.773857,164.230362,19123800,0.0,0.0,NaN,NaN
2024-08-23 00:00:00-04:00,165.282287,166.671625,164.399062,166.155579,14281600,0.0,0.0,NaN,NaN
2024-08-26 00:00:00-04:00,166.875073,168.090754,165.054048,166.651779,11990300,0.0,0.0,NaN,NaN
2024-08-27 00:00:00-04:00,166.334189,166.964350,164.895229,165.113556,13718200,0.0,0.0,NaN,NaN
2024-08-28 00:00:00-04:00,165.510508,166.115865,162.037149,163.247864,15208700,0.0,0.0,NaN,NaN
2024-08-29 00:00:00-04:00,164.796007,166.354064,160.749044,162.156250,17133800,0.0,0.0,NaN,NaN
2024-08-30 00:00:00-04:00,162.970016,164.021945,162.166184,163.853241,18498800,0.0,0.0,NaN,NaN
2024-09-03 00:00:00-04:00,162.071930,162.136438,156.653482,157.402740,26533100,0.0,0.0,NaN,NaN
2024-09-04 00:00:00-04:00,156.871793,159.179093,156.241632,156.608810,17410700,0.0,0.0,NaN,NaN


In [11]:
df_bb['bb_minimo'] = indicador_bb.bollinger_lband()
df_bb.head(30)

,Open,High,Low,Close,Volume,Dividends,Stock Splits,bb_minimo,bb_maximo
Date,,,,,,,,,
2024-08-22 00:00:00-04:00,167.753329,168.130441,163.773857,164.230362,19123800,0.0,0.0,NaN,NaN
2024-08-23 00:00:00-04:00,165.282287,166.671625,164.399062,166.155579,14281600,0.0,0.0,NaN,NaN
2024-08-26 00:00:00-04:00,166.875073,168.090754,165.054048,166.651779,11990300,0.0,0.0,NaN,NaN
2024-08-27 00:00:00-04:00,166.334189,166.964350,164.895229,165.113556,13718200,0.0,0.0,NaN,NaN
2024-08-28 00:00:00-04:00,165.510508,166.115865,162.037149,163.247864,15208700,0.0,0.0,NaN,NaN
2024-08-29 00:00:00-04:00,164.796007,166.354064,160.749044,162.156250,17133800,0.0,0.0,NaN,NaN
2024-08-30 00:00:00-04:00,162.970016,164.021945,162.166184,163.853241,18498800,0.0,0.0,NaN,NaN
2024-09-03 00:00:00-04:00,162.071930,162.136438,156.653482,157.402740,26533100,0.0,0.0,NaN,NaN
2024-09-04 00:00:00-04:00,156.871793,159.179093,156.241632,156.608810,17410700,0.0,0.0,NaN,NaN


In [12]:
# Seleciona apenas as colunas relevantes para exibição
df_bb = df_bb[['Close', 'bb_maximo', 'bb_minimo']]
df_bb.head(30)

,Close,bb_maximo,bb_minimo
Date,,,
2024-08-22 00:00:00-04:00,164.230362,NaN,NaN
2024-08-23 00:00:00-04:00,166.155579,NaN,NaN
2024-08-26 00:00:00-04:00,166.651779,NaN,NaN
2024-08-27 00:00:00-04:00,165.113556,NaN,NaN
2024-08-28 00:00:00-04:00,163.247864,NaN,NaN
2024-08-29 00:00:00-04:00,162.156250,NaN,NaN
2024-08-30 00:00:00-04:00,163.853241,NaN,NaN
2024-09-03 00:00:00-04:00,157.402740,NaN,NaN
2024-09-04 00:00:00-04:00,156.608810,NaN,NaN


In [13]:
# Calcula o objeto MACD para o preço de fechamento
macd_object = MACD(df_dados['Close'])
macd_object.macd()

Date
2024-08-22 00:00:00-04:00         NaN
2024-08-23 00:00:00-04:00         NaN
2024-08-26 00:00:00-04:00         NaN
2024-08-27 00:00:00-04:00         NaN
2024-08-28 00:00:00-04:00         NaN
                               ...   
2026-08-05 00:00:00-04:00    0.784210
2026-08-06 00:00:00-04:00    1.123895
2026-08-07 00:00:00-04:00    1.125940
2026-08-10 00:00:00-04:00    1.303770
2026-08-11 00:00:00-04:00    0.403965
Name: MACD_12_26, Length: 493, dtype: float64

In [14]:
# Calcula o indicador RSI para o preço de fechamento
rsi = RSIIndicator(df_dados.Close).rsi()
rsi

Date
2024-08-22 00:00:00-04:00          NaN
2024-08-23 00:00:00-04:00          NaN
2024-08-26 00:00:00-04:00          NaN
2024-08-27 00:00:00-04:00          NaN
2024-08-28 00:00:00-04:00          NaN
                               ...    
2026-08-05 00:00:00-04:00    54.265051
2026-08-06 00:00:00-04:00    52.550753
2026-08-07 00:00:00-04:00    50.993821
2026-08-10 00:00:00-04:00    52.142666
2026-08-11 00:00:00-04:00    45.868862
Name: rsi, Length: 493, dtype: float64

In [15]:
# Calcula a Média Móvel Simples (SMA) para o preço de fechamento
sma = SMAIndicator(df_dados.Close, window=14).sma_indicator()
sma

Date
2024-08-22 00:00:00-04:00           NaN
2024-08-23 00:00:00-04:00           NaN
2024-08-26 00:00:00-04:00           NaN
2024-08-27 00:00:00-04:00           NaN
2024-08-28 00:00:00-04:00           NaN
                                ...    
2026-08-05 00:00:00-04:00    344.016429
2026-08-06 00:00:00-04:00    344.766429
2026-08-07 00:00:00-04:00    344.916430
2026-08-10 00:00:00-04:00    345.605715
2026-08-11 00:00:00-04:00    345.683572
Name: sma_14, Length: 493, dtype: float64

In [16]:
# Calcula a Média Móvel Exponencial (EMA) para o preço de fechamento
ema = EMAIndicator(df_dados.Close).ema_indicator()
ema

Date
2024-08-22 00:00:00-04:00           NaN
2024-08-23 00:00:00-04:00           NaN
2024-08-26 00:00:00-04:00           NaN
2024-08-27 00:00:00-04:00           NaN
2024-08-28 00:00:00-04:00           NaN
                                ...    
2026-08-05 00:00:00-04:00    350.936083
2026-08-06 00:00:00-04:00    351.693938
2026-08-07 00:00:00-04:00    351.930747
2026-08-10 00:00:00-04:00    352.451980
2026-08-11 00:00:00-04:00    351.191716
Name: ema_14, Length: 493, dtype: float64